Before writing any Spark code you need a working environment. There are two paths: a **local PySpark installation** on your machine for development and learning, or **Databricks** for a fully managed cloud experience. This notebook walks through both.

## Two Setup Paths

![Setup Options](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-setup-options.svg)

Both paths give you a `SparkSession` — the code you write is identical. The difference is where the cluster runs and how much you have to configure.

## Option A — Local PySpark

### Prerequisites

PySpark requires **Java 8 or 11** on your PATH. Check with:

```bash
java -version
```

If Java is missing, install [Adoptium Temurin 11](https://adoptium.net) (free, cross-platform).

### Install PySpark

```bash
pip install pyspark          # latest stable
pip install pyspark==3.5.0   # pin to a specific version
```

That's it — PySpark bundles Spark itself, so no separate Spark download is needed.

### Optional extras

```bash
pip install pyspark[sql]     # Spark SQL extras
pip install delta-spark      # Delta Lake support
pip install jupyter          # if you want notebook support
```

In [ ]:
# Verify your PySpark installation
import pyspark
print(f"PySpark version: {pyspark.__version__}")

import os, subprocess
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java:", result.stderr.splitlines()[0] if result.stderr else result.stdout)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LocalSetup")
    .master("local[*]")              # use all available CPU cores
    .config("spark.sql.shuffle.partitions", "4")  # sensible default for local
    .getOrCreate()
)

print(f"Spark {spark.version} running on {spark.sparkContext.master}")
print(f"Cores available: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

### PySpark Interactive Shell

PySpark ships with an interactive shell that pre-creates `spark` and `sc` for you:

```bash
pyspark                        # Python shell
pyspark --master local[4]      # 4 cores
```

To use PySpark inside **Jupyter**, set the environment variables before launching:

```bash
export PYSPARK_DRIVER_PYTHON=jupyter
export PYSPARK_DRIVER_PYTHON_OPTS='notebook'
pyspark
```

Or just `import pyspark` and build a `SparkSession` manually inside a regular Jupyter notebook, as shown above.

## Option B — Databricks

Databricks is the company founded by the creators of Spark. Their platform runs Spark on cloud infrastructure (AWS, Azure, GCP) and adds Delta Lake, Unity Catalog, MLflow, and a collaborative notebook UI on top.

### Getting Started — Community Edition (Free)

1. Sign up at [community.cloud.databricks.com](https://community.cloud.databricks.com) — no credit card required
2. Create a **cluster**: Compute → Create Cluster → choose a Databricks Runtime (DBR) version → Create
3. Create a **notebook**: Workspace → Create → Notebook → attach to your cluster
4. A `spark` session is **already available** — no setup code needed

> Community Edition gives you a single-node cluster that terminates after 2 hours of inactivity. It is sufficient for learning all exam topics.

## Databricks Runtime (DBR)

The Databricks Runtime is a pre-configured environment that includes a pinned version of Spark, Delta Lake, popular Python libraries, and performance optimizations.

| DBR version | Spark version | Notes |
|---|---|---|
| DBR 14.x | Spark 3.5 | Latest LTS — recommended for exam prep |
| DBR 13.x | Spark 3.4 | Previous LTS |
| DBR 12.x | Spark 3.3 | Older LTS |
| DBR ML | Same + ML libs | Adds TensorFlow, PyTorch, XGBoost |

For the **Databricks Certified Associate Developer** exam, use DBR 13.x or 14.x.

## Databricks Notebook Basics

### Magic Commands

Databricks notebooks support magic commands that switch the language or behaviour of a cell:

| Magic | Effect |
|---|---|
| `%python` | Python cell (default) |
| `%sql` | Run a SQL query, results shown as a table |
| `%scala` | Scala cell |
| `%r` | R cell |
| `%md` | Markdown cell |
| `%sh` | Shell command on the driver |
| `%fs` | Shorthand for `dbutils.fs` commands |
| `%run ./other_notebook` | Execute another notebook inline |

### `display()` vs `show()`

On Databricks, prefer `display(df)` over `df.show()` — it renders an interactive table with sorting, filtering, and built-in charting. `show()` still works but outputs plain text.

## Databricks Utilities — `dbutils`

`dbutils` is a Databricks-specific utility object (not available in plain PySpark).

| Namespace | Common uses |
|---|---|
| `dbutils.fs` | Browse, copy, move, delete files in DBFS or cloud storage |
| `dbutils.secrets` | Read secrets from Databricks Secret Scopes (no hardcoded credentials) |
| `dbutils.widgets` | Create interactive input widgets in notebooks |
| `dbutils.notebook` | Run other notebooks, pass parameters, chain workflows |
| `dbutils.jobs` | Get job context (run ID, task name) when running as a job |

In [ ]:
# dbutils is only available on Databricks — this cell is for reference
# Uncomment and run inside a Databricks notebook

# List files in the Databricks File System root
# dbutils.fs.ls("/")

# List the built-in sample datasets
# dbutils.fs.ls("/databricks-datasets")

# Create a text widget
# dbutils.widgets.text("input_date", "2024-01-01", "Date")
# date_val = dbutils.widgets.get("input_date")

print("dbutils is Databricks-only — run this cell in a Databricks notebook")

## Databricks File System (DBFS)

DBFS is an abstraction layer over cloud object storage (S3, ADLS, GCS) mounted at `/` inside Databricks. It lets you use file-path syntax instead of cloud URIs in most places.

| Path style | Example |
|---|---|
| DBFS path | `/databricks-datasets/nyctaxi/tripdata/` |
| `dbfs:` URI | `dbfs:/user/data/myfile.parquet` |
| Cloud URI | `s3://my-bucket/data/` |

On **Unity Catalog** clusters (DBR 12+), prefer Unity Catalog Volume paths (`/Volumes/catalog/schema/volume/`) over DBFS for new data.

## Local vs Databricks — Key Differences

| | Local PySpark | Databricks |
|---|---|---|
| **Setup** | `pip install pyspark` + Java | Browser only, no install |
| **SparkSession** | Build with `.builder` | Pre-created as `spark` |
| **Cluster size** | Your laptop | Cloud VMs, auto-scaling |
| **Data access** | Local files, JDBC | DBFS, S3, ADLS, GCS, Unity Catalog |
| **`display()`** | Not available | Interactive table + charts |
| **`dbutils`** | Not available | Full utility suite |
| **Delta Lake** | `pip install delta-spark` + config | Built-in, default format |
| **Collaboration** | Git-based | Real-time multi-user notebooks |
| **Cost** | Free | Compute cost (Community Edition: free) |

In [ ]:
# Detect whether we're running locally or on Databricks
def is_databricks():
    try:
        dbutils  # noqa: F821
        return True
    except NameError:
        return False

if is_databricks():
    print("Running on Databricks")
    # spark is already available
else:
    print("Running locally")
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .appName("MyApp")
        .master("local[*]")
        .getOrCreate()
    )

print(f"Spark {spark.version}")

In [ ]:
# Smoke test — works identically on local and Databricks
df = spark.createDataFrame(
    [
        ("CUST0001", "Aarav Sharma",  "Mumbai"),
        ("CUST0002", "Priya Nair",    "Delhi"),
        ("CUST0003", "Rohan Gupta",   "Bengaluru"),
    ],
    ["customer_id", "full_name", "city"]
)
df.show()


## Summary

- **Local PySpark**: `pip install pyspark` (Java 8/11 required) → build a `SparkSession` with `.master("local[*]")`. Good for offline development.
- **Databricks**: sign up → create a cluster with a DBR version → `spark` is pre-created in every notebook. Good for cloud-scale work and exam practice.
- Databricks adds `display()`, `dbutils`, DBFS, magic commands (`%sql`, `%md`, `%run`), and Delta Lake out of the box.
- Use `spark.sql.shuffle.partitions = 4` locally to avoid the 200-partition default on small datasets.
- The Spark code you write is the same on both — only the session setup and file paths differ.